見つからない

In [1]:
import random
import numpy as np
import sys
from scipy.sparse import csr_matrix, hstack, vstack

# --- インポートの一本化と表示設定 ---
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

class GirthOptimizerV4:
    def __init__(self, P=768, J=3, L_half=6):
        self.P = P
        self.J = J
        self.L_half = L_half
        self.mid = P // 2
        # 基底（歯車）の生成
        self.rho_A = self._generate_random_cycle(range(0, self.mid))
        self.rho_B = self._generate_random_cycle(range(self.mid, self.P))
        self.identity = tuple(range(P))

    def _generate_random_cycle(self, r):
        indices = list(r)
        random.shuffle(indices)
        p = list(range(self.P))
        for i in range(len(indices)):
            p[indices[i]] = indices[(i + 1) % len(indices)]
        return tuple(p)

    def compose(self, p1, p2):
        return tuple(p1[p2[i]] for i in range(self.P))

    def get_power(self, base_p, k):
        res = self.identity
        curr = base_p
        k %= self.mid
        while k > 0:
            if k % 2 == 1: res = self.compose(res, curr)
            curr = self.compose(curr, curr)
            k //= 2
        return res

    def evaluate_hx_c4(self, current_F, current_G):
        """現在の F, G の一部を使用して Hx を仮組みし、C4 をカウントする"""
        # 未決定のブロックは Identity で埋める
        full_F = current_F + [self.identity] * (self.L_half - len(current_F))
        full_G = current_G + [self.identity] * (self.L_half - len(current_G))
        
        # 仮の Hx を構築
        Hx, _ = build_matrices_internal(full_F, full_G, self.P, self.J)
        
        # 4-cycle の高速計算 (int64)
        H_int = Hx.astype(np.int64)
        Gram = (H_int @ H_int.T).toarray()
        np.fill_diagonal(Gram, 0)
        return np.sum(Gram * (Gram - 1)) // 4

    def optimize(self):
        print(f"Starting Sequential Optimization for Hx (P={self.P}, J={self.J})")
        F_opt = []
        G_opt = []

        # 1. F ブロックを一つずつ決定
        for i in range(self.L_half):
            best_k = 0
            min_c4 = float('inf')
            # 候補となる指数をサンプリングして試行
            candidates = random.sample(range(1, self.mid), min(30, self.mid-1))
            for k in candidates:
                test_f = self.get_power(self.rho_A, k)
                c4 = self.evaluate_hx_c4(F_opt + [test_f], G_opt)
                if c4 < min_c4:
                    min_c4 = c4
                    best_k = k
                if min_c4 == 0: break
            F_opt.append(self.get_power(self.rho_A, best_k))
            print(f"Fixed F[{i}] (k={best_k}). Min C4 in partial Hx: {min_c4}")

        # 2. G ブロックを一つずつ決定
        for j in range(self.L_half):
            best_k = 0
            min_c4 = float('inf')
            candidates = random.sample(range(1, self.mid), min(30, self.mid-1))
            for k in candidates:
                test_g = self.get_power(self.rho_B, k)
                c4 = self.evaluate_hx_c4(F_opt, G_opt + [test_g])
                if c4 < min_c4:
                    min_c4 = c4
                    best_k = k
                if min_c4 == 0: break
            G_opt.append(self.get_power(self.rho_B, best_k))
            print(f"Fixed G[{j}] (k={best_k}). Min C4 in partial Hx: {min_c4}")

        return F_opt, G_opt

def tuple_to_sparse(p, size):
    rows = np.arange(size)
    cols = np.array(p)
    return csr_matrix((np.ones(size, dtype=np.int8), (rows, cols)), shape=(size, size))

def build_matrices_internal(F, G, P, J):
    L_h = len(F)
    F_m = [tuple_to_sparse(f, P) for f in F]
    G_m = [tuple_to_sparse(g, P) for g in G]
    hx_rows = []
    hz_rows = []
    for i in range(J):
        row_x = [F_m[(j-i)%L_h] for j in range(L_h)] + [G_m[(j-i)%L_h] for j in range(L_h)]
        row_z = [G_m[(i-j)%L_h].transpose() for j in range(L_h)] + [F_m[(i-j)%L_h].transpose() for j in range(L_h)]
        hx_rows.append(hstack(row_x))
        hz_rows.append(hstack(row_z))
    return vstack(hx_rows), vstack(hz_rows)

def count_cycles_direct(H):
    num_checks, num_vars = H.shape
    adj_c = [H.getrow(i).indices for i in range(num_checks)]
    H_csc = H.tocsc()
    adj_v = [H_csc.getcol(j).indices for j in range(num_vars)]
    c4, c6 = 0, 0
    # 4-cycle
    shared = {}
    for c1 in range(num_checks):
        for v in adj_c[c1]:
            for c2 in adj_v[v]:
                if c2 > c1:
                    shared[(c1, c2)] = shared.get((c1, c2), 0) + 1
    for count in shared.values():
        if count >= 2: c4 += count * (count - 1) // 2
    # 6-cycle
    for c1 in range(num_checks):
        for v1 in adj_c[c1]:
            for c2 in adj_v[v1]:
                if c2 <= c1: continue
                for v2 in adj_c[c2]:
                    if v2 == v1: continue
                    for c3 in adj_v[v2]:
                        if c3 <= c1 or c3 == c2: continue
                        common = set(adj_c[c3]) & set(adj_c[c1])
                        for v3 in common:
                            if v3 != v1 and v3 != v2: c6 += 1
    return c4, c6 // 2

# --- 実行 ---
P, J = 768, 3
optimizer = GirthOptimizerV4(P=P, J=J)
F_final, G_final = optimizer.optimize()

# 最終確認
Hx, Hz = build_matrices_internal(F_final, G_final, P, J)
print(f"\nFinal Check - Hx Shape: {Hx.shape}")
# 直交性確認
ortho = (Hx @ Hz.transpose())
ortho.data %= 2
ortho.eliminate_zeros()
print(f"CSS Condition Violation: {ortho.nnz}")

c4, c6 = count_cycles_direct(Hx)
print(f"C4: {c4}, C6: {c6}")

Starting Sequential Optimization for Hx (P=768, J=3)
Fixed F[0] (k=63). Min C4 in partial Hx: 127872
Fixed F[1] (k=153). Min C4 in partial Hx: 114432
Fixed F[2] (k=325). Min C4 in partial Hx: 105600
Fixed F[3] (k=284). Min C4 in partial Hx: 97920
Fixed F[4] (k=34). Min C4 in partial Hx: 93312


KeyboardInterrupt: 